# ANALYSE-04 : Trois primitives de PFR, et l'endroit exact où elles cessent de valoir

**Série** : SymbolicAI / Lean — Digestions de résultats profonds, companion du capstone [Lean-20 — Digestions Tao](../Lean-20-Capstone-Digestions-Tao-Python.ipynb)
**Source** : ANALYSE-03-PFR-Lean.ipynb (digestion PFR par Gowers-Green-Manners-Tao, 2023)
**Lac source** : https://github.com/teorth/pfr (formalisation collaborative Lean 4)
**Kernel** : `.net-csharp` — le notebook conceptuel C# illustre les primitives, les preuves formelles restent dans ANALYSE-03
**Jumeau Python** : [Version Python](ANALYSE-04-PFR-Primitives-Python.ipynb)
**Jumeau Rust** : [ANALYSE-04-PFR-Primitives-Rust](ANALYSE-04-PFR-Primitives-Rust.ipynb) — même marche, kernel `rust` (evcxr), std seul.

## Navigation

| Notebook précédent | Notebook suivant |
|---|---|
| [ANALYSE-03 - PFR Entropy Method](ANALYSE-03-PFR-Lean.ipynb) | [ANALYSE-05 - KLS](ANALYSE-05-KLS-Lean-Python.ipynb) |

***

## Pourquoi ce notebook

La digestion ANALYSE-03 a établi l'essentiel : PFR est un théorème de « compression ⇒ structure », et l'entropie y est **monnaie commune** entre combinatoire additive, machines causales et ICT. Ce qu'ANALYSE-04 ajoute n'est pas une reprise de la preuve — c'est l'**extraction de primitives transportables** et, indissociablement, **le test de leurs limites**.

Trois primitives, et **une seule** est inconditionnellement transportable :

| # | Primitive | Transport |
|---|---|---|
| 1 | Distance entropique de Ruzsa — `d[X;Y] = H(X′−Y′) − ½H(X′) − ½H(Y′)` | **conditionnel** : exige une structure de groupe additif |
| 2 | Décomposition projection / fibres — `X ⟶ π(X) + X\|π(X)` | large : règle de chaîne + information mutuelle conditionnelle |
| 3 | La fonctionnelle τ — descente strictement monotone ; quand elle ne peut plus descendre, **la structure est forcée** | patron, sous hypothèses à expliciter |

## Tester les limites des primitives

> **Une primitive qui ne transfère pas est un résultat.**

La distance de Ruzsa exige une structure de groupe additif : des proxys **sans structure de groupe** ne justifient pas son application. Le notebook cherche donc **où ça casse**, aussi bien que là où ça marche.

## Honnêteté de portée

> PFR ne « prouve » évidemment rien pour ICT — son résultat est spécifique à la combinatoire additive, avec des propriétés particulières de `F₂ⁿ` jusque dans son endgame. **Mais il fournit un exemple CERTIFIÉ où une quantité informationnelle FORCE une structure algébrique. C'est beaucoup plus précieux qu'une analogie.**

Et sur `τ_strictly_decreases` : c'est un **analogue de fonction de Lyapunov pour la preuve**, pas une Lyapunov dynamique au sens physique. La qualification doit figurer dans le notebook.

## Plan

1. Primitive 1 — Distance entropique de Ruzsa : énoncé, calcul, **test de limite**.
2. Primitive 2 — Décomposition projection / fibres : règle de chaîne, **test de limite**.
3. Primitive 3 — La fonctionnelle τ : patron de descente, **test de limite**.
4. Conclusion — lien vers ANALYSE-03 et le lac `pfr`.

**Suite de la sous-série** : [ANALYSE-06b — Sensibilité 2026 (notre lac face aux familles d'openai/math)](ANALYSE-06b-Sensibilite-2026-Lean.ipynb)
**Suite de la sous-série** : [ANALYSE-07 — Percolation critique (notre lac face aux familles 2026 213/236)](ANALYSE-07-Percolation-Critique.ipynb)

## 1. Primitive 1 — Distance entropique de Ruzsa

### 1.1 Énoncé

Soient `X`, `Y` deux variables aléatoires sur un groupe abélien `(G, +)`. On note `X′` une copie indépendante de `X` et `Y′` une copie indépendante de `Y`. La **distance de Ruzsa** est définie par :

```
d[X;Y] = H(X′ − Y′) − ½ H(X′) − ½ H(Y′)
```

où `H` est l'entropie de Shannon en bits.

**Invariants** (sur groupe abélien) :
- `d[X;Y] ≥ 0` toujours (inégalité de conditional mutual information / chaîne).
- `d[X;Y] = d[Y;X]` (symétrique).
- **Translation commune** : `d[X+a; Y+a] = d[X;Y]` pour tout `a` déterministe — l'invariant inconditionnel.
- `d[X;X] = 0` **si et seulement si `X` est uniforme sur un translaté d'un sous-groupe** (en particulier, uniforme sur `G` tout entier — `G` est lui-même le translaté `0 + G`) ; pour une loi quelconque, seule `d[X;X] ≥ 0` est garantie — le test 4 mesure un `d[X;X] = 0.5` bit et le test 5 la nullité sur le translaté `{1,3} = 1 + {0,2}`, qui n'est **pas** un sous-groupe.
- **Triangle** : `d[X;Z] ≤ d[X;Y] + d[Y;Z]`.

**Sémantique** : `d[X;Y]` mesure combien « les distributions sont loin d'être des translatées d'un même sous-groupe uniforme ».

### 1.2 Calcul sur son terrain — un groupe additif fini

On prend `G = ℤ/nℤ` et on compare deux variables `X`, `Y` à distributions connues.

In [1]:
// 1.2 — Distance de Ruzsa sur un groupe additif fini
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

string Inv(FormattableString text) => text.ToString(CultureInfo.InvariantCulture);

// Marge pour le seul bruit d'arrondi en double sur les identités et les petites lois exactes.
const double NumericalTolerance = 1e-9;
var parityChecks = new List<(string Section, string Label, double Error)>();

void CheckNear(string section, string label, double actual, double expected,
    double tolerance = NumericalTolerance)
{
    var error = Math.Abs(actual - expected);
    var ok = double.IsFinite(error) && error <= tolerance;
    var verdict = ok ? "CONFORME" : "ÉCART";
    Console.WriteLine(Inv($"  {verdict} — {label} : calculé = {actual:G17}, attendu = {expected:G17}, écart = {error:G3}"));
    if (!ok)
        throw new InvalidOperationException(Inv($"{section} / {label} : tolérance {tolerance:G3} dépassée."));
    parityChecks.Add((section, label, error));
}

void CheckCondition(string section, string label, bool actual, bool expected = true)
{
    var ok = actual == expected;
    var verdict = ok ? "CONFORME" : "ÉCART";
    Console.WriteLine(Inv($"  {verdict} — {label} : observé = {actual}, attendu = {expected}"));
    if (!ok)
        throw new InvalidOperationException($"{section} / {label} : verdict inattendu.");
    // Écart catégoriel : zéro quand le verdict observé est celui attendu.
    parityChecks.Add((section, label, 0.0));
}

// Entropie de Shannon en bits à partir des masses de probabilité.
double EntropyBits(IEnumerable<double> probabilities) =>
    -probabilities.Where(p => p > 0.0).Sum(p => p * Math.Log2(p));

int Mod(int value, int n) => ((value % n) + n) % n;

// Image de la loi produit : additionner les masses des couples qui ont la même image.
Dictionary<int, double> OperationLaw(IReadOnlyDictionary<int, double> xDist,
    IReadOnlyDictionary<int, double> yDist, Func<int, int, int> operation)
{
    var result = new Dictionary<int, double>();
    foreach (var x in xDist)
    foreach (var y in yDist)
    {
        int key = operation(x.Key, y.Key);
        result.TryGetValue(key, out double mass);
        result[key] = mass + x.Value * y.Value;
    }
    return result;
}

double RuzsaDistanceForOperation(IReadOnlyDictionary<int, double> xDist,
    IReadOnlyDictionary<int, double> yDist, Func<int, int, int> operation)
{
    var difference = OperationLaw(xDist, yDist, operation);
    return EntropyBits(difference.Values) - 0.5 * EntropyBits(xDist.Values) - 0.5 * EntropyBits(yDist.Values);
}

double RuzsaDistance(IReadOnlyDictionary<int, double> xDist,
    IReadOnlyDictionary<int, double> yDist, int n)
{
    // X' - Y' : convolution signée modulo n
    return RuzsaDistanceForOperation(xDist, yDist, (x, y) => Mod(x - y, n));
}

// Les lois et les valeurs de référence sont citées du carnet Python, cellule 2.
const string sectionRuzsa = "Ruzsa — tests 1 à 5";

// Test 1 : X et Y uniformes indépendants sur ℤ/8ℤ
const int n = 8;
var uniform = Enumerable.Range(0, n).ToDictionary(k => k, _ => 1.0 / n);
var test1 = RuzsaDistance(uniform, uniform, n);
Console.WriteLine(Inv($"Test 1 : X = Y uniformes i.i.d. sur ℤ/{n}ℤ"));
Console.WriteLine(Inv($"  d[X;Y] = {test1:F4}  (attendu ~ 0)"));
CheckNear(sectionRuzsa, "Test 1 : d[X;Y]", test1, 0.0);

// Test 2 : Y = X + 1 (translation)
var shifted = Enumerable.Range(0, n).ToDictionary(k => k, k => uniform[Mod(k - 1, n)]);
var test2 = RuzsaDistance(uniform, shifted, n);
Console.WriteLine("Test 2 : Y = X + 1 (translation)");
Console.WriteLine(Inv($"  d[X;Y] = {test2:F4}  (attendu 0)"));
CheckNear(sectionRuzsa, "Test 2 : d[X;Y]", test2, 0.0);

// Test 3 : X uniforme sur ℤ/8ℤ, Y dégénéré en 0 (H(Y) = 0).
// RuzsaDistance ne reçoit que les lois MARGINALES et convole des copies
// indépendantes X', Y' : aucun argument de couplage n'entre dans ce calcul.
// Ici X' - Y' = X', donc d[X;Y] = H(X) - ½H(X) = ½H(X) = 1.5 bits.
var yDegenerate = Enumerable.Range(0, n).ToDictionary(k => k, k => k == 0 ? 1.0 : 0.0);
var test3 = RuzsaDistance(uniform, yDegenerate, n);
var hDegenerate = EntropyBits(yDegenerate.Values);
Console.WriteLine("Test 3 : X uniforme, Y concentré en 0");
Console.WriteLine(Inv($"  d[X;Y] = {test3:F4}  (H(Y)={hDegenerate:F0}, formule dégénère)"));
CheckNear(sectionRuzsa, "Test 3 : H(Y)", hDegenerate, 0.0);
CheckNear(sectionRuzsa, "Test 3 : d[X;Y]", test3, 1.5);

// Test 4 : contre-exemple -- d[X;X] = 0 exige l'uniformité de X sur un
// translaté d'un sous-groupe (le test 5 montre qu'un translaté suffit).
// X = {0: 0.5, 1: 0.5} sur Z/4Z : X' - X'' vaut 0 avec proba 0.5,
// et 1, 3 avec proba 0.25 chacun -> H(X'-X'') = 1.5 bits, H(X) = 1 bit,
// donc d[X;X] = 1.5 - 1 = 0.5 bit, PAS 0.
var xNonCoset = new Dictionary<int, double> { [0] = 0.5, [1] = 0.5 };
var test4 = RuzsaDistance(xNonCoset, xNonCoset, 4);
Console.WriteLine("Test 4 : X = {0: 0.5, 1: 0.5} sur Z/4Z -- loi NON uniforme sur le groupe");
Console.WriteLine(Inv($"  d[X;X] = {test4:F4}  (attendu 0.5 bit : {{0,1}} n'est pas un translaté de sous-groupe)"));
CheckNear(sectionRuzsa, "Test 4 : d[X;X]", test4, 0.5);

// L'invariant inconditionnel est la translation COMMUNE : d[X+a; Y+a] = d[X;Y].
foreach (int a in new[] { 1, 2, 3 })
{
    var xTranslated = xNonCoset.ToDictionary(kv => Mod(kv.Key + a, 4), kv => kv.Value); // loi de X + a
    var dTranslated = RuzsaDistance(xTranslated, xTranslated, 4);
    Console.WriteLine(Inv($"  d[X+{a}; X+{a}] = {dTranslated:F4}  (translation commune : inchangé)"));
    CheckNear(sectionRuzsa, Inv($"Test 4 : translation de {a}, référence Python"), dTranslated, 0.5);
    CheckNear(sectionRuzsa, Inv($"Test 4 : translation de {a}, invariance"), dTranslated, test4);
}

// Cas favorable : uniforme sur Z/4Z (le sous-groupe entier G = 0 + G), la nullité est récupérée
var uniform4 = Enumerable.Range(0, 4).ToDictionary(k => k, _ => 0.25);
var test4bis = RuzsaDistance(uniform4, uniform4, 4);
Console.WriteLine(Inv($"Test 4bis : X uniforme sur Z/4Z -> d[X;X] = {test4bis:F4}  (nullité de l'uniforme)"));
CheckNear(sectionRuzsa, "Test 4bis : d[X;X]", test4bis, 0.0);

// Test 5 : témoin de nullité sur un translaté QUI N'EST PAS un sous-groupe.
// X = {1: 0.5, 3: 0.5} sur Z/4Z : support {1,3} = 1 + {0,2}, translaté du
// sous-groupe {0,2} -- il ne contient pas 0, ce n'est donc pas un sous-groupe.
// X' - X'' vaut 0 avec proba 0.5 et 2 avec proba 0.5 -> H(X'-X'') = H(X) = 1 bit,
// donc d[X;X] = 0 : la nullité vaut pour tout translaté, pas seulement les sous-groupes.
var xCoset = new Dictionary<int, double> { [1] = 0.5, [3] = 0.5 };
var test5 = RuzsaDistance(xCoset, xCoset, 4);
Console.WriteLine("Test 5 : X = {1: 0.5, 3: 0.5} sur Z/4Z -- uniforme sur le translaté 1+{0,2}, PAS un sous-groupe");
Console.WriteLine(Inv($"  d[X;X] = {test5:F4}  (nullité : uniforme sur un translaté de sous-groupe)"));
CheckNear(sectionRuzsa, "Test 5 : d[X;X]", test5, 0.0);

Test 1 : X = Y uniformes i.i.d. sur ℤ/8ℤ


  d[X;Y] = 0.0000  (attendu ~ 0)


  CONFORME — Test 1 : d[X;Y] : calculé = 0, attendu = 0, écart = 0


Test 2 : Y = X + 1 (translation)


  d[X;Y] = 0.0000  (attendu 0)


  CONFORME — Test 2 : d[X;Y] : calculé = 0, attendu = 0, écart = 0


... (17 outputs left out)

  CONFORME — Test 4bis : d[X;X] : calculé = 0, attendu = 0, écart = 0


Test 5 : X = {1: 0.5, 3: 0.5} sur Z/4Z -- uniforme sur le translaté 1+{0,2}, PAS un sous-groupe


  d[X;X] = 0.0000  (nullité : uniforme sur un translaté de sous-groupe)


  CONFORME — Test 5 : d[X;X] : calculé = 0, attendu = 0, écart = 0


### Lecture du résultat

Les cinq tests illustrent les **invariants** sur groupe additif :
- Test 1 : variables i.i.d. **uniformes sur le même groupe** ⇒ `d[X;Y] ≈ 0` — la nullité tient à l'uniformité, pas à la seule identité des lois.
- Test 2 : translation d'une loi **uniforme** ⇒ `d[X;Y] = 0` exactement (la distance de Ruzsa est invariante par translation, c'est sa propriété fondamentale).
- Test 3 : dégénérescence — la formule reste définie mais perd sa sémantique ; `RuzsaDistance` ne voit que les lois marginales, jamais un couplage.
- Test 4 : **contre-exemple** — `X = {0: ½, 1: ½}` sur `ℤ/4ℤ` n'est pas uniforme sur le groupe, et `d[X;X] = 0.5` bit : l'invariant `d[X;X] = 0` ne vaut que pour `X` uniforme sur un **translaté de sous-groupe**. L'invariant inconditionnel est la translation **commune** `d[X+a; Y+a] = d[X;Y]`, vérifiée au même test ; une translation simple `X = a + Y` donne `d[X;Y] = d[Y;Y]`, nul seulement si `Y` est uniforme sur un **translaté de sous-groupe**.
- Test 5 : **témoin de nullité sur un translaté** — `X = {1: ½, 3: ½}` est uniforme sur le translaté `1 + {0,2}` (qui ne contient pas `0`, donc pas un sous-groupe) et `d[X;X] = 0` : le critère est bien l'uniformité sur un translaté, pas sur un sous-groupe seul. Le contraste (a) de la cellule suivante le rejoint : le même support `{0,1}` est un sous-groupe du groupe de Klein (`d = 0`) mais n'est le translaté d'aucun sous-groupe de `ℤ/4ℤ` (`d = 0.5`).

**Point clé** : la définition **exige** la convolution `X′ − Y′`, qui n'a de sens que sur groupe additif. C'est ce que « Ruzsa exige une structure de groupe additif » veut dire.

### 1.3 Test de limite — quand la primitive ne transporte plus

On garde la formule, les distributions et l'entropie **inchangées**, et on ne fait
varier que l'opération `x − y`. Deux exhibitions :

- **(a)** le même ensemble `{0,1,2,3}` muni de **deux lois de groupe différentes**
  (`ℤ/4ℤ` et le groupe de Klein) donne **deux valeurs différentes** de `d` pour les
  **mêmes** distributions — la valeur de `d` n'est donc pas une fonction des seules
  distributions, mais du couple (distributions, loi de groupe) ;
- **(b)** en retirant l'inversibilité de `x ↦ x − y`, `d` devient **négatif** — et une
  distance négative n'en est plus une.

In [2]:
// 1.3 — Test de limite : ce que la formule cesse de mesurer sans loi de groupe.
// On ne change QUE l'opération « x − y ». Le reste — les distributions, la
// formule, l'entropie — est identique à la cellule précédente.
const string sectionLimiteRuzsa = "Ruzsa — changement de loi";
const string sectionAleatoireRuzsa = "Ruzsa — signes des minima";

// (a) Même ensemble, mêmes distributions, deux lois de groupe différentes.
// X = Y = uniforme sur {0, 1}, plongées dans un ensemble à 4 éléments.
var xLimit = new Dictionary<int, double> { [0] = 0.5, [1] = 0.5 };
Func<int, int, int> z4 = (a, b) => Mod(a - b, 4); // ℤ/4ℤ : groupe cyclique
Func<int, int, int> klein = (a, b) => a ^ b; // (ℤ/2ℤ)² : groupe de Klein

var cyclicDifference = OperationLaw(xLimit, xLimit, z4);
var cyclicEntropy = EntropyBits(cyclicDifference.Values);
var dCyclic = RuzsaDistanceForOperation(xLimit, xLimit, z4);
var dKlein = RuzsaDistanceForOperation(xLimit, xLimit, klein);
Console.WriteLine("(a) mêmes distributions, deux lois de groupe sur le même ensemble {0,1,2,3}");
Console.WriteLine(Inv($"  ℤ/4ℤ  : H(X−Y) = {cyclicEntropy:F4} bits   →  d[X;Y] = {dCyclic:F4}"));
Console.WriteLine(Inv($"  Klein : d[X;Y] = {dKlein:F4}"));

// Masse théorique de la loi et valeurs déterministes citées de la sortie Python, cellule 4(a).
CheckNear(sectionLimiteRuzsa, "ℤ/4ℤ : masse de la convolution", cyclicDifference.Values.Sum(), 1.0);
CheckNear(sectionLimiteRuzsa, "ℤ/4ℤ : H(X−Y)", cyclicEntropy, 1.5);
CheckNear(sectionLimiteRuzsa, "ℤ/4ℤ : d[X;Y]", dCyclic, 0.5);
CheckNear(sectionLimiteRuzsa, "Klein : d[X;Y]", dKlein, 0.0);
Console.WriteLine("  La valeur de d n'est donc PAS une fonction des seules distributions :");
Console.WriteLine("  c'est une fonction du couple (distributions, loi de groupe).");

// (b) On retire l'inversibilité : « x − y » cesse d'être une bijection en x.
// Si d devient négatif, ce n'est plus une distance.
const int pairsPerOperation = 40_000;
const int citedPairsPerOperation = 40_000;
const int supportSize = 4;
var ruzsaRng = new Random(7);

// Gamma(forme 1, échelle 1) est une exponentielle de taux 1 : inversion de sa fonction de répartition.
double SampleGammaShapeOne(Random generator)
{
    double u;
    do { u = generator.NextDouble(); } while (u == 0.0);
    return -Math.Log(1.0 - u);
}

Dictionary<int, double> SampleDirichlet(Random generator, int size)
{
    var gamma = new double[size];
    for (int i = 0; i < size; i++)
        gamma[i] = SampleGammaShapeOne(generator);
    double total = gamma.Sum();
    return Enumerable.Range(0, size).ToDictionary(i => i, i => gamma[i] / total);
}

// Opérations, effectif et signes attendus cités de la cellule 4(b) du Python ; aucun minimum repris.
var ruzsaOperations = new (string Name, string Kind, Func<int, int, int> Apply, bool ExpectedNonnegative)[]
{
    ("(a − b) mod 4", "groupe cyclique", z4, true),
    ("a XOR b", "groupe de Klein", klein, true),
    ("max(a − b, 0)", "non inversible", (a, b) => Math.Max(a - b, 0), false),
    ("(a × b) mod 4", "non inversible", (a, b) => (a * b) % 4, false)
};

string pairCountText = pairsPerOperation.ToString("N0", CultureInfo.InvariantCulture).Replace(',', ' ');
Console.WriteLine(Inv($"\n(b) {pairCountText} couples de distributions tirés au hasard par opération sur {{0,1,2,3}} :"));
foreach (var law in ruzsaOperations)
{
    double minimum = double.PositiveInfinity;
    int measuredPairs = 0;
    for (int i = 0; i < pairsPerOperation; i++)
    {
        var p = SampleDirichlet(ruzsaRng, supportSize);
        var q = SampleDirichlet(ruzsaRng, supportSize);
        minimum = Math.Min(minimum, RuzsaDistanceForOperation(p, q, law.Apply));
        measuredPairs++;
    }

    CheckNear(sectionAleatoireRuzsa, $"{law.Name} : couples mesurés", measuredPairs, citedPairsPerOperation, 0.0);
    bool nonnegative = minimum >= 0.0;
    if (!double.IsFinite(minimum))
        throw new InvalidOperationException($"{law.Name} : minimum non fini.");
    string verdict = nonnegative ? "min d ≥ 0" : "min d < 0 : PLUS une distance";
    Console.WriteLine(Inv($"  {law.Name,-14} {law.Kind,-16} →  min d (C#) = {minimum:+0.0000;-0.0000;+0.0000}   {verdict}"));
    CheckCondition(sectionAleatoireRuzsa, $"{law.Name} : min d ≥ 0", nonnegative, law.ExpectedNonnegative);
}
Console.WriteLine("\n  Les deux opérations non inversibles rendent d négatif. Une « distance »");
Console.WriteLine("  négative n'en est pas une : la primitive a cessé de valoir, et on voit où.");

(a) mêmes distributions, deux lois de groupe sur le même ensemble {0,1,2,3}


  ℤ/4ℤ  : H(X−Y) = 1.5000 bits   →  d[X;Y] = 0.5000


  Klein : d[X;Y] = 0.0000


  CONFORME — ℤ/4ℤ : masse de la convolution : calculé = 1, attendu = 1, écart = 0


  CONFORME — ℤ/4ℤ : H(X−Y) : calculé = 1.5, attendu = 1.5, écart = 0


  CONFORME — ℤ/4ℤ : d[X;Y] : calculé = 0.5, attendu = 0.5, écart = 0


... (14 outputs left out)

  (a × b) mod 4  non inversible   →  min d (C#) = -0.9517   min d < 0 : PLUS une distance


  CONFORME — (a × b) mod 4 : min d ≥ 0 : observé = False, attendu = False



  Les deux opérations non inversibles rendent d négatif. Une « distance »


  négative n'en est pas une : la primitive a cessé de valoir, et on voit où.


### Verdict Primitive 1

**Transportable sous condition** : `RuzsaDistance` exige une **structure de groupe additif** `(G, +)`. La cellule précédente le montre en deux temps — la valeur de `d` **change** quand on change la loi de groupe à distributions fixées, et sans inversibilité de `x ↦ x − y` elle **passe sous zéro**, donc cesse d'être une distance. C'est exactement la classe « conditionnelle » du tableau d'introduction.

**Leçon** : la primitive a une **puissance incomparable** sur son terrain (groupes abéliens, `F₂ⁿ`, entropie appliquée à la combinatoire additive), mais l'**utiliser hors de ce terrain** serait du « mapping affine » — c'est-à-dire plaquer une structure sans la mériter. Le carnet ANALYSE-03 lui-même s'en garde : la preuve PFR ne s'applique qu'à `F₂ⁿ`, pas à un objet métrique général.

#### Parité avec le carnet Python — tirages de Dirichlet

Le carnet C# garde le support `{0,1,2,3}`, les quatre opérations et les 40 000 couples par opération. Il génère ses propres tirages avec `new Random(7)` et `SampleGammaShapeOne`, sans reproduire le générateur MT19937 ni l'échantillonneur gamma de Python. Les minima affichés sont ceux du C# : les deux premiers sont non négatifs, les deux derniers strictement négatifs. Le code compare ces quatre signes à ceux lus dans la sortie Python, pas les valeurs numériques des minima. Les signes observés sur les groupes illustrent la non-négativité sans la démontrer ; chacun des deux minima négatifs fournit un contre-exemple pour son opération.

### 1.4 Test 6 — l'inégalité triangulaire, mesurée

L'énoncé de 1.1 liste le **triangle** `d[X;Z] ≤ d[X;Y] + d[Y;Z]` parmi les invariants — c'est le seul de la liste qui n'était pas encore mesuré, alors qu'il n'est pas une évidence de la formule : l'inégalité triangulaire entropique de Ruzsa est un théorème du terrain, démontré par Tao, *Sumset and inverse sumset theory for Shannon entropy* (arXiv:0906.4387), théorème 1.10, équation (16), sous la forme `distR(X, Z) ≤ distR(X, Y) + distR(Y, Z)`. Le test suivant la mesure sur trois triplets de `ℤ/8ℤ` choisis pour discriminer — égalité exacte, saturation, triplet tendu — en réutilisant `RuzsaDistance` telle quelle, sans réimplémentation.

In [3]:
// 1.4 — Test 6 : l'inégalité triangulaire d[X;Z] <= d[X;Y] + d[Y;Z], mesurée.
// RuzsaDistance (cellule 1.2) est réutilisée telle quelle : trois distances par
// triplet, l'inégalité lue sur trois témoins qui discriminent.
const string sectionTriangle = "Ruzsa — triangle";
const int n8 = 8;
var subgroupA = new Dictionary<int, double> { [0] = 0.25, [2] = 0.25, [4] = 0.25, [6] = 0.25 }; // uniforme sur le sous-groupe A = {0,2,4,6}
var subgroupAPlus = new Dictionary<int, double> { [1] = 0.25, [3] = 0.25, [5] = 0.25, [7] = 0.25 }; // uniforme sur le translaté A + 1
var u8 = Enumerable.Range(0, n8).ToDictionary(k => k, _ => 1.0 / n8); // uniforme sur Z/8Z tout entier
var degen0 = Enumerable.Range(0, n8).ToDictionary(k => k, k => k == 0 ? 1.0 : 0.0); // dégénéré en 0

void ReadTriplet(int number, string name, IReadOnlyDictionary<int, double> x,
    IReadOnlyDictionary<int, double> y, IReadOnlyDictionary<int, double> z, int modulus,
    (double XY, double YZ, double XZ, double Bound, double Margin, bool Triangle) reference)
{
    double dxy = RuzsaDistance(x, y, modulus);
    double dyz = RuzsaDistance(y, z, modulus);
    double dxz = RuzsaDistance(x, z, modulus);
    double bound = dxy + dyz;
    double margin = bound - dxz;
    bool triangle = dxz <= bound + NumericalTolerance;
    string verdict = triangle ? "OK" : "VIOLEE";
    string label = Inv($"Triplet {number}");

    Console.WriteLine(Inv($"{label} -- {name}"));
    Console.WriteLine(Inv($"  d[X;Y] = {dxy:F4}   d[Y;Z] = {dyz:F4}   d[X;Z] = {dxz:F4}"));
    Console.WriteLine(Inv($"  inégalité : {dxz:F4} <= {bound:F4}   marge = {margin:+0.0000;-0.0000;+0.0000} bit   {verdict}"));
    CheckNear(sectionTriangle, $"{label} : d[X;Y]", dxy, reference.XY);
    CheckNear(sectionTriangle, $"{label} : d[Y;Z]", dyz, reference.YZ);
    CheckNear(sectionTriangle, $"{label} : d[X;Z]", dxz, reference.XZ);
    CheckNear(sectionTriangle, $"{label} : borne", bound, reference.Bound);
    CheckNear(sectionTriangle, $"{label} : marge", margin, reference.Margin);
    CheckCondition(sectionTriangle, $"{label} : inégalité triangulaire", triangle, reference.Triangle);
}

// Triplets et valeurs attendues cités de la cellule 7 du Python.
Console.WriteLine("Test 6 : l'inégalité triangulaire d[X;Z] <= d[X;Y] + d[Y;Z], sur Z/8Z");
Console.WriteLine();
ReadTriplet(1, "égalité exacte : X = Y = Z uniformes sur A = {0,2,4,6}",
    subgroupA, subgroupA, subgroupA, n8, (0.0, 0.0, 0.0, 0.0, 0.0, true));
Console.WriteLine();
ReadTriplet(2, "saturation sans colinéarité : X, Y sur A ; Z sur A + 1",
    subgroupA, subgroupA, subgroupAPlus, n8, (0.0, 0.0, 0.0, 0.0, 0.0, true));
Console.WriteLine();
ReadTriplet(3, "tendu : Y dégénéré en 0, X uniforme sur Z/8Z, Z sur A",
    u8, degen0, subgroupA, n8, (1.5, 1.0, 0.5, 2.5, 2.0, true));

Test 6 : l'inégalité triangulaire d[X;Z] <= d[X;Y] + d[Y;Z], sur Z/8Z


Triplet 1 -- égalité exacte : X = Y = Z uniformes sur A = {0,2,4,6}


  d[X;Y] = 0.0000   d[Y;Z] = 0.0000   d[X;Z] = 0.0000


  inégalité : 0.0000 <= 0.0000   marge = +0.0000 bit   OK


  CONFORME — Triplet 1 : d[X;Y] : calculé = 0, attendu = 0, écart = 0


  CONFORME — Triplet 1 : d[Y;Z] : calculé = 0, attendu = 0, écart = 0


... (18 outputs left out)

  CONFORME — Triplet 3 : d[X;Z] : calculé = 0.5, attendu = 0.5, écart = 0


  CONFORME — Triplet 3 : borne : calculé = 2.5, attendu = 2.5, écart = 0


  CONFORME — Triplet 3 : marge : calculé = 2, attendu = 2, écart = 0


  CONFORME — Triplet 3 : inégalité triangulaire : observé = True, attendu = True


### Lecture du résultat — Test 6

- **Triplet 1** (`X = Y = Z` uniformes sur le sous-groupe `A = {0,2,4,6}`) : les trois distances valent `0` — l'inégalité est une **égalité exacte** (`0 ≤ 0 + 0`).
- **Triplet 2** (`X`, `Y` sur `A` ; `Z` sur le translaté `A + 1`) : les trois distances valent encore `0` alors que `X` et `Z` n'ont **pas la même loi** — l'inégalité est **saturée sans colinéarité** des lois. C'est le phénomène du test 2 : deux uniformes sur des translatés d'un même sous-groupe sont à distance nulle, et la borne ne peut pas descendre plus bas.
- **Triplet 3** (`Y` dégénéré en `0` entre `X` uniforme sur `ℤ/8ℤ` et `Z` uniforme sur `A`) : `d[X;Y] = ½H(X) = 1.5` bit et `d[Y;Z] = ½H(Z) = 1.0` bit — la borne vaut `2.5` bits, et la mesure rend `d[X;Z] = 0.5` bit : l'inégalité tient avec une **marge de 2.0 bits**. Un test qui ne rendrait que `0.5 ≤ 2.5` n'apprendrait rien ; les trois témoins ensemble disent où l'inégalité colle et où elle respire.

Le contrôle négatif est déjà dans le carnet : le test de limite (b) de 1.3 montre que sans inversibilité de `x ↦ x − y`, `d` devient **négatif** — la triangularité meurt avec la structure de groupe, comme les autres invariants. La ligne d'énoncé du triangle en 1.1 est désormais adossée à sa mesure.

### Exercice 1 — Ruzsa sur deux groupes de même ordre : Z/4Z contre (Z/2Z)²

La cellule du cours calcule `d[X;Y]` sur Z/8Z (`n = 8`). Le verdict ci-dessus dit que la valeur de `d` **change quand on change la loi de groupe, à distributions fixées** — vérifiez-le sur deux groupes d'ordre 4 construits sur les mêmes étiquettes {0, 1, 2, 3} :

- **Z/4Z** (cyclique) : `addZ4(x, y) = (x + y) % 4` ;
- **(Z/2Z)²** (groupe de Klein) : `addKlein(x, y) = x ^ y` (ou exclusif bit à bit sur les étiquettes codées en 2 bits).

Pour les **mêmes** distributions — X uniforme sur {0, 1} et Y uniforme sur {2, 3} — calculez `d[X;Y] = H(X−Y) − ½H(X) − ½H(Y)` sous **chacune des deux lois**, et comparez les deux valeurs.

In [4]:
// Illustration 1 — Exercice 1 : d[X;Y] sur Z/4Z puis (Z/2Z)^2, mêmes étiquettes {0,1,2,3} (à compléter)
Dictionary<int, double> ConvolutionExercice(
    IReadOnlyDictionary<int, double> px, IReadOnlyDictionary<int, double> py,
    Func<int, int, int> add)
{
    // TODO etudiant (Etape 1)
    return new Dictionary<int, double>();
}

double DistanceRuzsaGroupeExercice(
    IReadOnlyDictionary<int, double> px, IReadOnlyDictionary<int, double> py,
    Func<int, int, int> add, Func<int, int> inverse)
{
    // TODO etudiant (Etape 2)
    return 0.0;
}

// Etape 3 : X uniforme sur {0,1}, Y uniforme sur {2,3}, deux lois de groupe -- comparer.
Console.WriteLine("Exercice à compléter : d[X;Y] sous Z/4Z puis sous (Z/2Z)^2, mêmes X et Y.");

Exercice à compléter : d[X;Y] sous Z/4Z puis sous (Z/2Z)^2, mêmes X et Y.


## 2. Primitive 2 — Décomposition projection / fibres

### 2.1 Énoncé

Soient `X` une variable aléatoire et `Y` une variable auxiliaire. On note
`π(X) := 𝔼[X | Y]` (espérance conditionnelle L2) ou plus généralement une
**projection** sur une σ-algèbre. La **décomposition informationnelle** est :

```
X = π(X) + (X − π(X))
```

où `X − π(X)` désigne le **résidu** (composante inexpliquée par la
projection). Cette écriture est un raccourci de présentation : la
primitive est **informationnelle** (entropies), pas numérique — sur des
v.a. continues le « + » n'a pas de sens au sens usuel.

**Invariants** (règle de chaîne) :

- `H(X) = H(π(X)) + H(X | π(X))` — la chaîne informationnelle, universelle.
- `I(X ; Y) = H(X) − H(X | Y)` — définition de l'information mutuelle.
- `𝔼[X − M | M] = 0` — **le résidu est d'espérance conditionnelle nulle
  par rapport à la projection L2** `M := 𝔼[X | Y]` (régression).
  On note `P = g(X)` une projection générale (pas L2) : `P` n'a aucune
  raison de satisfaire `𝔼[X − P | P] = 0`. Le Cas 2 ci-dessous utilise
  `P = 𝟙{X ≥ 15}` (indicatrice binaire) et mesure des moyennes du résidu
  non nulles et différentes entre les fibres `high` et `low` : le seuil
  ne restitue pas la valeur de la note. Pour la projection L2,
  l'orthogonalité implique `Cov(X − M, M) = 0`, sans garantir à elle seule
  l'indépendance. Pour l'indicatrice, la covariance et l'information
  mutuelle entre le résidu et la projection sont toutes deux non nulles.
- `I(X ; π(X)) = H(π(X))` — et **non 0** en général : lorsque `π(X)` est une
  fonction déterministe de `X`, toute l'information de `π(X)` vient de `X`.

**Erreur à éviter** : l'affirmation `I(X ; π(X)) = 0` est **fausse** dès
que `π(X)` est une fonction déterministe non constante de `X`. Pour une
projection L2, si le résidu et la projection sont conjointement gaussiens,
leur décorrélation implique leur indépendance. C'est alors
`I(X − π(X) ; π(X))` qui s'annule, **pas** `I(X ; π(X))` en général.

### 2.2 Application : décomposer un dataset ML

In [5]:
// Primitive 2 — Décomposition projection / fibres sur un cas concret
// On considère deux scénarios, qui testent des identités DIFFÉRENTES :
//   Cas 1 : projection sur Y (cours). Y n'est PAS une fonction de X,
//           donc on ne peut pas écrire H(X) = H(π(X)) + H(X | π(X))
//           avec π(X) := Y. La chaîne correcte est H(X) = I(X;Y) + H(X|Y),
//           et l'écart H(Y) − I(X;Y) > 0 mesure l'information que Y porte
//           sur autre chose que X.
//   Cas 2 : projection π(X) = 𝟙{X ≥ 15}. π EST une fonction de X,
//           donc on peut écrire H(X) = H(π(X)) + H(X | π(X)) ET
//           I(X ; π(X)) = H(π(X)) ≠ 0.

const string sectionProjection = "Projection / fibres — notes";
// La référence Python est arrondie au millième : un demi-millième au plus,
// avec vérification supplémentaire de l'affichage exact au format F3.
const double PythonF3Tolerance = 0.0005;

void CheckPythonF3(string section, string label, double actual, double cited)
{
    CheckNear(section, label + " (référence Python citée)", actual, cited, PythonF3Tolerance);
    string shown = actual.ToString("F3", CultureInfo.InvariantCulture);
    string expected = cited.ToString("F3", CultureInfo.InvariantCulture);
    if (!string.Equals(shown, expected, StringComparison.Ordinal))
        throw new InvalidOperationException($"{section} / {label} : affiché {shown}, attendu {expected}.");
}

// Normaliser les effectifs d'un échantillon pour obtenir les masses de sa loi empirique.
double[] EmpiricalProbabilities<T>(IEnumerable<T> sample)
{
    int[] counts = sample.GroupBy(value => value).Select(group => group.Count()).ToArray();
    int total = counts.Sum();
    return counts.Select(count => (double)count / total).ToArray();
}

double EmpiricalEntropy<T>(IEnumerable<T> sample) => EntropyBits(EmpiricalProbabilities(sample));

double ConditionalGradeEntropy<TKey>(IEnumerable<IGrouping<TKey, int>> fibres, int total) =>
    fibres.Sum(fibre => (double)fibre.Count() / total * EmpiricalEntropy(fibre));

// Données citées : les 25 paires (cours, note) de la cellule 12 du carnet Python.
var gradeData = new (string Course, int Grade)[]
{
    ("Math", 14), ("Math", 16), ("Math", 12), ("Math", 18), ("Math", 15),
    ("Phys", 11), ("Phys", 13), ("Phys", 9),  ("Phys", 12), ("Phys", 14),
    ("Info", 17), ("Info", 19), ("Info", 16), ("Info", 18), ("Info", 20),
    ("Bio",  10), ("Bio",  12), ("Bio",  11), ("Bio",  13), ("Bio",  9),
    ("Chim", 13), ("Chim", 15), ("Chim", 14), ("Chim", 16), ("Chim", 12)
};
int gradeCount = gradeData.Length;
double hGrade = EmpiricalEntropy(gradeData.Select(row => row.Grade));

// === Cas 1 — projection sur Y (cours). Y n'est PAS fonction de X. ===
var courseFibres = gradeData.GroupBy(row => row.Course, row => row.Grade).ToArray();
double hGradeGivenCourse = ConditionalGradeEntropy(courseFibres, gradeCount);
double hCourse = EmpiricalEntropy(gradeData.Select(row => row.Course));
double iGradeCourse = hGrade - hGradeGivenCourse;
double courseGap = hCourse - iGradeCourse;
double courseChain = hGradeGivenCourse + iGradeCourse;
bool courseIsFunctionOfGrade = gradeData.GroupBy(row => row.Grade)
    .All(fibre => fibre.Select(row => row.Course).Distinct().Count() == 1);

Console.WriteLine($"=== Cas 1 : projection sur Y (cours). Y {(courseIsFunctionOfGrade ? "est" : "n'est PAS")} fonction de X. ===");
Console.WriteLine(Inv($"  H(X)        = {hGrade:F3} bits"));
Console.WriteLine(Inv($"  H(X | Y)    = {hGradeGivenCourse:F3} bits  (variation inexpliquée par le cours)"));
Console.WriteLine(Inv($"  H(Y)        = {hCourse:F3} bits"));
Console.WriteLine(Inv($"  I(X;Y)      = H(X) - H(X|Y) = {iGradeCourse:F3} bits  (info que le cours porte sur la note)"));
Console.WriteLine(Inv($"  Écart H(Y) − I(X;Y) = {courseGap:F3} bits"));
Console.WriteLine(Inv($"    → I(X;Y) = {iGradeCourse:F3} ≠ H(Y) = {hCourse:F3} : Y porte {courseGap:F3} bits"));
Console.WriteLine("    d'information qui ne concernent pas X (témoin numérique de l'écart).");
Console.WriteLine("    C'est précisément pour ça qu'on ne peut PAS écrire");
Console.WriteLine("    H(X) = H(Y) + H(X|Y) : ce serait supposer Y = π(X) (fonction de X).");
Console.WriteLine(Inv($"  Chaîne vérifiée : H(X) = {hGrade:F3} ≈ H(X|Y) + I(X;Y) = {courseChain:F3}"));

// Valeurs attendues citées de la sortie Python, cellule 12 (cas 1).
CheckPythonF3(sectionProjection, "H(X)", hGrade, 3.433);
CheckPythonF3(sectionProjection, "H(X | Y)", hGradeGivenCourse, 2.322);
CheckPythonF3(sectionProjection, "H(Y)", hCourse, 2.322);
CheckPythonF3(sectionProjection, "I(X;Y)", iGradeCourse, 1.111);
CheckPythonF3(sectionProjection, "H(Y) − I(X;Y)", courseGap, 1.211);
CheckPythonF3(sectionProjection, "H(X | Y) + I(X;Y)", courseChain, 3.433);
CheckNear(sectionProjection, "Chaîne avec Y", courseChain, hGrade);
CheckCondition(sectionProjection, "Y est fonction de X", courseIsFunctionOfGrade, false);
CheckCondition(sectionProjection, "H(Y) − I(X;Y) > 0", courseGap > NumericalTolerance);

// === Cas 2 — projection π(X) = 𝟙{X ≥ 15}. π est fonction de X. ===
const int gradeThreshold = 15; // Seuil cité du carnet Python.
var projectedGrades = gradeData.Select(row => (row.Grade, P: row.Grade >= gradeThreshold ? 1 : 0)).ToArray();
var gradeFibres = projectedGrades.GroupBy(row => row.P, row => row.Grade).ToArray();
double hIndicator = EmpiricalEntropy(projectedGrades.Select(row => row.P));
double hGradeGivenIndicator = ConditionalGradeEntropy(gradeFibres, gradeCount);
double iGradeIndicator = hGrade - hGradeGivenIndicator;
double indicatorChain = hIndicator + hGradeGivenIndicator;
bool indicatorIdentity = Math.Abs(iGradeIndicator - hIndicator) < NumericalTolerance;

Console.WriteLine();
Console.WriteLine(Inv($"=== Cas 2 : projection π(X) = 𝟙{{X ≥ {gradeThreshold}}}. π est fonction de X. ==="));
Console.WriteLine(Inv($"  H(π(X))     = {hIndicator:F3} bits  (entropie de l'indicatrice 'note ≥ {gradeThreshold}')"));
Console.WriteLine(Inv($"  H(X | π(X)) = {hGradeGivenIndicator:F3} bits  (variation intra-fibre)"));
Console.WriteLine(Inv($"  I(X ; π(X)) = H(X) - H(X | π(X)) = {iGradeIndicator:F3} bits"));
Console.WriteLine(Inv($"  I(X ; π(X)) = H(π(X)) ? {indicatorIdentity}  →  I(X;π(X)) = {iGradeIndicator:F3}"));
Console.WriteLine("    ≢ 0 bits.");
Console.WriteLine("    Ici π(X) est une fonction de X, donc l'identité correcte est");
Console.WriteLine("    I(X ; π(X)) = H(π(X)) : la projection porte TOUTE son information");
Console.WriteLine("    de X, et l'écart à 0 mesure exactement le contenu informationnel de π.");
Console.WriteLine(Inv($"  Chaîne vérifiée : H(X) = {hGrade:F3} ≈ H(π(X)) + H(X | π(X)) = {indicatorChain:F3}"));

// === Témoins directs des propriétés du résidu (section 2.1) ===
// Résidu r = X − π_num(X) où π_num = 1 si X ≥ 15, 0 sinon. La prose
// des sections 2.1 et 2.2 cite I(X−π(X); π(X)) ≈ 0.78 bit et Cov ≈ 0.98
// — on les calcule ici sur les 25 paires du carnet, qui sont la base
// de toutes les autres mesures (donc falsifiables par le lecteur).
// Ces deux témoins mesurent que π = 𝟙{X≥15} n'est PAS la projection L2 :
// l'orthogonalité E[X−π(X)|π(X)] = 0 est garantie pour la projection L2 (régression).
// Ici le résidu n'est ni indépendant (I ≠ 0) ni même décorrélé
// (Cov ≠ 0) — la projection binaire capte l'ordre mais pas la valeur.
var residualPairs = projectedGrades.Select(row => (R: row.Grade - row.P, row.P)).ToArray();
double hResidual = EmpiricalEntropy(residualPairs.Select(row => row.R));
double hResidualIndicator = EmpiricalEntropy(residualPairs);
double iResidualIndicator = hResidual + hIndicator - hResidualIndicator;
double meanResidual = residualPairs.Average(row => row.R);
double meanIndicator = residualPairs.Average(row => row.P);
double covResidualIndicator = residualPairs.Average(row => (row.R - meanResidual) * (row.P - meanIndicator));
Console.WriteLine(Inv($"  I(X−π(X); π(X)) = {iResidualIndicator:F3} bits"));
Console.WriteLine(Inv($"  Cov(X−π(X), π(X)) = {covResidualIndicator:F3}"));

// === Témoin E[X−π(X) | π(X)] = 0 ? Garantie pour la projection L2 M = 𝔼[X|·] ===
// Pour la projection P = 𝟙{X≥15} (Cas 2), l'orthogonalité NE TIENT PAS :
// E[X−P | P=high] et E[X−P | P=low] diffèrent et ne sont pas nulles.
// Ces moyennes sont calculées sur les mêmes paires que I et Cov.
double meanResidualHigh = residualPairs.Where(row => row.P == 1).Average(row => row.R);
double meanResidualLow = residualPairs.Where(row => row.P == 0).Average(row => row.R);
Console.WriteLine(Inv($"  E[X−π(X) | π(X)=high] = {meanResidualHigh:F3}  (témoin : ≠ 0)"));
Console.WriteLine(Inv($"  E[X−π(X) | π(X)=low]  = {meanResidualLow:F3}  (témoin : ≠ 0)"));
Console.WriteLine(Inv($"  → π = 𝟙{{X≥{gradeThreshold}}} n'est PAS la projection L2 M = 𝔼[X|·] :"));
Console.WriteLine("    l'orthogonalité 𝔼[X−M | M] = 0 est garantie pour M, pas pour une projection générale P.");

// Valeurs attendues citées de la sortie Python, cellule 12 (cas 2 et résidus).
CheckPythonF3(sectionProjection, "H(π(X))", hIndicator, 0.971);
CheckPythonF3(sectionProjection, "H(X | π(X))", hGradeGivenIndicator, 2.462);
CheckPythonF3(sectionProjection, "I(X;π(X))", iGradeIndicator, 0.971);
CheckPythonF3(sectionProjection, "H(π(X)) + H(X | π(X))", indicatorChain, 3.433);
CheckPythonF3(sectionProjection, "I(X−π(X);π(X))", iResidualIndicator, 0.777);
CheckPythonF3(sectionProjection, "Cov(X−π(X), π(X))", covResidualIndicator, 0.976);
CheckPythonF3(sectionProjection, "E[X−π(X) | π(X)=high]", meanResidualHigh, 16.000);
CheckPythonF3(sectionProjection, "E[X−π(X) | π(X)=low]", meanResidualLow, 11.933);
CheckNear(sectionProjection, "Chaîne avec π(X)", indicatorChain, hGrade);
CheckCondition(sectionProjection, "I(X;π(X)) = H(π(X))", indicatorIdentity);
CheckCondition(sectionProjection, "I(X;π(X)) ≠ 0", iGradeIndicator > NumericalTolerance);
CheckCondition(sectionProjection, "I(X−π(X);π(X)) ≠ 0", iResidualIndicator > NumericalTolerance);
CheckCondition(sectionProjection, "Cov(X−π(X), π(X)) ≠ 0", Math.Abs(covResidualIndicator) > NumericalTolerance);
CheckCondition(sectionProjection, "E[X−π(X) | π(X)=high] ≠ 0", Math.Abs(meanResidualHigh) > NumericalTolerance);
CheckCondition(sectionProjection, "E[X−π(X) | π(X)=low] ≠ 0", Math.Abs(meanResidualLow) > NumericalTolerance);

=== Cas 1 : projection sur Y (cours). Y n'est PAS fonction de X. ===


  H(X)        = 3.433 bits


  H(X | Y)    = 2.322 bits  (variation inexpliquée par le cours)


  H(Y)        = 2.322 bits


  I(X;Y)      = H(X) - H(X|Y) = 1.111 bits  (info que le cours porte sur la note)


  Écart H(Y) − I(X;Y) = 1.211 bits


... (41 outputs left out)

  CONFORME — I(X−π(X);π(X)) ≠ 0 : observé = True, attendu = True


  CONFORME — Cov(X−π(X), π(X)) ≠ 0 : observé = True, attendu = True


  CONFORME — E[X−π(X) | π(X)=high] ≠ 0 : observé = True, attendu = True


  CONFORME — E[X−π(X) | π(X)=low] ≠ 0 : observé = True, attendu = True


### Lecture du résultat

Deux scénarios, deux identités :

- **Cas 1 (Y n'est pas fonction de X)** : la règle de chaîne
  `H(X) = H(X|Y) + I(X;Y)` est vérifiée numériquement. L'écart
  `H(Y) − I(X;Y) ≈ 1.21 bits` est le **témoin** que Y porte de
  l'information sur autre chose que X — c'est précisément ce qui
  empêche d'écrire `H(X) = H(Y) + H(X|Y)`. Ici, parler de
  « projection `π(X) := Y` » n'a aucun sens, puisque Y n'est pas
  une fonction de X.
- **Cas 2 (π est fonction de X)** : la chaîne
  `H(X) = H(π(X)) + H(X | π(X))` tient, et `I(X ; π(X)) = H(π(X))`
  (et **non 0**).
  `H(π(X))` capture « ce que la projection capte » (le bit
  d'appartenance à la fibre `high` ou `low`) ; `H(X | π(X))`
  capture « ce qui reste dans les fibres » (la variation de note
  à l'intérieur d'une même fibre).

**Sémantique opérationnelle** :
- `H(π(X))` ≈ « ce que la projection capte » (la partition qu'elle définit).
- `H(X | π(X))` ≈ « ce qui reste dans les fibres » (la variation intra-fibre).
- `I(X ; π(X)) = H(π(X))` ≠ 0 en général : la projection **porte** de
  l'information sur X. L'identité `𝔼[X − π(X) | π(X)] = 0` n'est pas
  universelle — elle ne tient **que** pour la projection L2
  `M := 𝔼[X | ·]` (orthogonalité par construction de l'espérance
  conditionnelle, cf. section 2.1, invariants). Pour une projection
  générale `P = g(X)` — ex. `P = 𝟙{X ≥ 15}` du Cas 2 — les témoins
  `𝔼[X − P | P = high] ≈ 16.0` et `𝔼[X − P | P = low] ≈ 11.93`
  (section 2.2) montrent qu'elle ne tient pas, et l'indépendance
  `I(X − π(X) ; π(X)) = 0` n'est elle-même garantie que dans le cas
  gaussien. Cette lecture prolonge la section 2.2 : `π(X) = Y` n'est
  pas une *projection* au sens de la primitive 2, c'est une variable
  auxiliaire (Cas 1) ; la primitive 2 ne s'applique proprement qu'à
  un `π` qui est une fonction de X (Cas 2).

### 2.3 Test de limite — quand la primitive ne transporte plus

La règle de chaîne **H(X) = H(X|Y) + I(X;Y)** est-elle valide pour
**n'importe quelle** « projection » `Y` ?

In [6]:
// 2.3 — Test de limite : du texte brut à une loi empirique
// Test de limite : projection arbitraire sur un objet non-probabiliste
// On prend un texte brut et on essaie de définir H(X) comme nombre de mots uniques.
// Mais sans distribution de probabilité, ce n'est PAS de l'entropie de Shannon.
const string sectionText = "Projection / fibres — texte brut";
const string rawText = "le chat mange le poisson le chien mange la viande"; // Texte cité de la cellule 14 du carnet Python.
var words = rawText.Split(' ', StringSplitOptions.RemoveEmptyEntries);
int uniqueWordCount = words.Distinct(StringComparer.Ordinal).Count();
int totalWordCount = words.Length;
Console.WriteLine("Entrée citée du carnet Python :");
Console.WriteLine($"Texte : '{rawText}'");
Console.WriteLine(Inv($"Mots uniques : {uniqueWordCount}, Mots totaux : {totalWordCount}"));

// H_naive = -sum(1/N * log2(1/N)) sur N = nb total mots ? NON : ce n'est pas une distribution sur les mots distincts.
// La primitive EXIGE une distribution de probabilité sur X et Y.
// Sur du texte brut, on a un COMPTAGE, pas une distribution : il faut normaliser.
// Construire une distribution : P(mot) = count / total
var wordProbabilities = EmpiricalProbabilities(words);
double hText = EntropyBits(wordProbabilities);
Console.WriteLine(Inv($"Distribution normalisée → H(text) = {hText:F3} bits"));

// Effectifs et entropie attendus cités de la sortie Python, cellule 14.
CheckNear(sectionText, "Mots uniques (référence Python citée)", uniqueWordCount, 7, 0.0);
CheckNear(sectionText, "Mots totaux (référence Python citée)", totalWordCount, 10, 0.0);
CheckPythonF3(sectionText, "H(text)", hText, 2.646);
CheckNear(sectionText, "Masse de la loi empirique", wordProbabilities.Sum(), 1.0);

Console.WriteLine("Une fois normalisée, l'entropie a un sens. Mais :");
Console.WriteLine("  - Pas de σ-algèbre canonique sur 'mots'");
Console.WriteLine("  - 'Y' = 'premier mot de la phrase' ? — Y est-il une v.a. légitime ?");
Console.WriteLine("  - Sans mesure de probabilité, 'projection' n'a pas de sens formel");
Console.WriteLine();
Console.WriteLine("Verdict Primitive 2 :");
Console.WriteLine("  Décomposition X = π(X) + (X|π(X)) exige DISTRIBUTION DE PROBABILITÉ.");
Console.WriteLine("  Transportable LARGE (la règle de chaîne est universelle en théorie de l'information).");
Console.WriteLine("  Mais 'projection' et 'fibres' doivent être formalisées en proba —");
Console.WriteLine("  sinon la décomposition est un formalisme vide.");

Entrée citée du carnet Python :


Texte : 'le chat mange le poisson le chien mange la viande'


Mots uniques : 7, Mots totaux : 10


Distribution normalisée → H(text) = 2.646 bits


  CONFORME — Mots uniques (référence Python citée) : calculé = 7, attendu = 7, écart = 0


  CONFORME — Mots totaux (référence Python citée) : calculé = 10, attendu = 10, écart = 0


... (7 outputs left out)

  Décomposition X = π(X) + (X|π(X)) exige DISTRIBUTION DE PROBABILITÉ.


  Transportable LARGE (la règle de chaîne est universelle en théorie de l'information).


  Mais 'projection' et 'fibres' doivent être formalisées en proba —


  sinon la décomposition est un formalisme vide.


### Verdict Primitive 2

**Transportable large** : la **règle de chaîne** `H(X) = H(X|Y) + I(X;Y)` est **universelle en théorie de l'information** — elle n'exige qu'un espace probabilisé `(Ω, ℱ, ℙ)` et deux v.a. `X`, `Y`. C'est la primitive la plus transportable des trois.

**Mais** : pour qu'elle ait une **sémantique opérationnelle**, il faut que la projection `π(X) := 𝔼[X | Y]` soit **bien définie** — c'est-à-dire que `Y` soit une variable aléatoire légitime (mesurable, à valeurs dans un espace convenable). Sur du texte brut, du graphe, ou tout objet sans mesure canonique, la décomposition est un formalisme vide.

**Leçon** : la primitive est **plus transportable que Ruzsa**, mais elle exige quand même un cadre probabiliste. Le mapping « projection/fibres » vers ICT (où « fibres » = classes d'équivalence, « projection » = quotient) est **légitime** SI on construit explicitement la mesure.

### Exercice 2 — Décomposer un autre dataset en projection / fibres

Le cours décompose un dataset ML par projection sur une coordonnée puis étudie les fibres. Refaites l'exercice sur le **dataset jouet des notes** : 8 étudiants `(maths, physique)`, projection sur `maths >= 10`, fibres = les deux groupes. Mesurez pour chaque fibre : effectif, moyenne `physique`, écart-type.

**Données citées du carnet Python** : `[(12, 14), (8, 6), (15, 11), (9, 13), (11, 10), (6, 5), (14, 16), (10, 9)]`.

In [7]:
// Illustration 2 — Exercice 2 : décomposition projection / fibres sur les notes (à compléter)
(IReadOnlyList<(int Maths, int Physique)> Admis, IReadOnlyList<(int Maths, int Physique)> Refuses)
    ProjeterNotesExercice(IReadOnlyList<(int Maths, int Physique)> points, int seuil = 10)
{
    // TODO etudiant — stub pédagogique.
    return (Array.Empty<(int Maths, int Physique)>(), Array.Empty<(int Maths, int Physique)>());
}

(int Effectif, double Moyenne, double EcartType)
    StatistiquesFibreExercice(IReadOnlyList<(int Maths, int Physique)> fibre)
{
    // TODO etudiant — stub pédagogique.
    return (0, 0.0, 0.0);
}

// Données citées du carnet Python, cellule 17.
Console.WriteLine("Exercice à compléter : décomposer [(12, 14), (8, 6), (15, 11), (9, 13), (11, 10), (6, 5), (14, 16), (10, 9)] (données citées du carnet Python).");

Exercice à compléter : décomposer [(12, 14), (8, 6), (15, 11), (9, 13), (11, 10), (6, 5), (14, 16), (10, 9)] (données citées du carnet Python).


## 3. Primitive 3 — La fonctionnelle τ

### 3.1 Énoncé

Soit `S` une structure combinatoire munie d'une fonctionnelle `τ : S → ℝ⁺` qui décroît **strictement** à chaque étape d'une procédure de raffinement :

```
S₀ ⊃ S₁ ⊃ S₂ ⊃ ...
```

avec `τ(Si₊₁) < τ(Si)`. Quand `τ` **ne peut plus descendre** (`τ(Si₊₁) = τ(Si)`), la procédure s'arrête et la **structure est forcée**.

**Sémantique** : `τ` est un **analogue de fonction de Lyapunov pour la preuve**. C'est l'invariant qui garantit la terminaison.

### 3.2 Patron : descente monotone sur un problème jouet

On prend le problème « factoriser un entier `N` » avec `τ(N) = nombre de diviseurs premiers distincts`. La procédure : tester la divisibilité par 2, 3, 5, 7, 11, ... jusqu'à `√N`.

In [8]:
// Primitive 3 — τ = nombre de diviseurs premiers distincts (omega(N)), sur N = 60.
// Procédure : à chaque étape, retirer toute la valuation du plus petit facteur premier.
const string sectionOmega = "τ — factorisation";
const double ExactIntegerTolerance = 0.0; // Les valeurs comparées sont des entiers exacts.

int Omega(int value)
{
    int remainder = value;
    int count = 0;
    // p <= remainder / p équivaut à p² <= remainder sans risque de débordement.
    for (int p = 2; p <= remainder / p; p++)
    {
        if (remainder % p != 0)
            continue;
        count++;
        while (remainder % p == 0)
            remainder /= p;
    }
    if (remainder > 1)
        count++;
    return count;
}

string FormatIntegerSequence(IEnumerable<int> values) =>
    "[" + string.Join(", ", values.Select(value => value.ToString(CultureInfo.InvariantCulture))) + "]";

string FormatBooleanSequence(IEnumerable<bool> values) =>
    "[" + string.Join(", ", values.Select(value => value ? "True" : "False")) + "]";

bool[] StrictDecreaseFlags(IEnumerable<int> values) =>
    values.Zip(values.Skip(1), (previous, next) => next < previous).ToArray();

// Trace : N₀ = 60, on retire 2² → 15, puis 3 → 5, puis 5 → 1.
// Chaque étape élimine un facteur premier distinct :
// ω(60)=3, ω(15)=2, ω(5)=1, ω(1)=0.
const int omegaInput = 60; // Entrée citée de la cellule 19 du carnet Python.
int omegaInitial = Omega(omegaInput);
Console.WriteLine(Inv($"ω({omegaInput}) = {omegaInitial} (2×2×3×5 → facteurs premiers distincts 2, 3 et 5 ; factorisation citée)"));

// Re-vérification : 60 = 2² × 3 × 5 → 3 facteurs premiers distincts.
const int omegaManualReference = 3; // Vérification manuelle citée de la sortie Python, cellule 19.
Console.WriteLine(Inv($"Vérif manuelle citée du carnet Python : 60 = 2² × 3 × 5 → ω = {omegaManualReference}"));

// La procédure de raffinement s'arrête à N=1, où ω=0.
int omegaRemainder = omegaInput;
var omegaSteps = new List<int> { omegaRemainder };
while (omegaRemainder > 1)
{
    int divisor = 2;
    while (divisor <= omegaRemainder / divisor && omegaRemainder % divisor != 0)
        divisor++;

    if (divisor > omegaRemainder / divisor)
    {
        // N est premier : retirer son unique facteur premier restant.
        omegaRemainder = 1;
    }
    else
    {
        while (omegaRemainder % divisor == 0)
            omegaRemainder /= divisor;
    }
    omegaSteps.Add(omegaRemainder);
}

int[] omegaValues = omegaSteps.Select(Omega).ToArray();
bool[] omegaStrict = StrictDecreaseFlags(omegaValues);
string omegaStepsText = FormatIntegerSequence(omegaSteps);
string omegaValuesText = FormatIntegerSequence(omegaValues);
string omegaStrictText = FormatBooleanSequence(omegaStrict);
Console.WriteLine($"Séquence de raffinement : {omegaStepsText}");
Console.WriteLine($"ω à chaque étape : {omegaValuesText}");
Console.WriteLine($"ω décroît-elle strictement ? {omegaStrictText}");

// Références citées de la sortie Python, cellule 19 : ordre, longueur et valeurs doivent coïncider.
int[] citedOmegaSteps = { 60, 15, 5, 1 };
int[] citedOmegaValues = { 3, 2, 1, 0 };
bool[] citedOmegaStrict = { true, true, true };
CheckNear(sectionOmega, "ω(60) — vérification manuelle citée", omegaInitial, omegaManualReference, ExactIntegerTolerance);
CheckCondition(sectionOmega,
    $"Séquence de raffinement : calculée {omegaStepsText}, référence citée {FormatIntegerSequence(citedOmegaSteps)}",
    omegaSteps.SequenceEqual(citedOmegaSteps));
CheckCondition(sectionOmega,
    $"Valeurs de ω : calculées {omegaValuesText}, référence citée {FormatIntegerSequence(citedOmegaValues)}",
    omegaValues.SequenceEqual(citedOmegaValues));
CheckCondition(sectionOmega,
    $"Décroissance stricte : calculée {omegaStrictText}, référence citée {FormatBooleanSequence(citedOmegaStrict)}",
    omegaStrict.SequenceEqual(citedOmegaStrict));

ω(60) = 3 (2×2×3×5 → facteurs premiers distincts 2, 3 et 5 ; factorisation citée)


Vérif manuelle citée du carnet Python : 60 = 2² × 3 × 5 → ω = 3


Séquence de raffinement : [60, 15, 5, 1]


ω à chaque étape : [3, 2, 1, 0]


ω décroît-elle strictement ? [True, True, True]


  CONFORME — ω(60) — vérification manuelle citée : calculé = 3, attendu = 3, écart = 0


  CONFORME — Séquence de raffinement : calculée [60, 15, 5, 1], référence citée [60, 15, 5, 1] : observé = True, attendu = True


  CONFORME — Valeurs de ω : calculées [3, 2, 1, 0], référence citée [3, 2, 1, 0] : observé = True, attendu = True


  CONFORME — Décroissance stricte : calculée [True, True, True], référence citée [True, True, True] : observé = True, attendu = True


### Lecture du résultat

La séquence `60 → 15 → 5 → 1` montre `ω` **strictement décroissant** jusqu'à 0. La procédure **termine** parce que `ω` est un entier positif et ne peut pas descendre en dessous de 0.

**Point clé** : c'est exactement le rôle de `τ_strictly_decreases` dans ANALYSE-03 (lac `pfr`) — c'est **l'invariant qui garantit la terminaison** de la procédure de preuve. Sans cet invariant strictement décroissant, on ne pourrait pas conclure que la procédure s'arrête.

### Exercice 3 — Trouver le τ d'Euclide

Le patron du cours exige une fonctionnelle `τ` qui décroît **strictement** à chaque étape d'une procédure, sur un ordre bien fondé. Le cas positif le plus ancien : **l'algorithme d'Euclide** `(a, b) ↦ (b, a mod b)`.

Pour plusieurs couples `(a, b)`, construisez la trajectoire des paires, proposez une fonctionnelle `τ` et vérifiez que la suite de ses valeurs est **strictement décroissante** et atteint 0.

**Couples cités du carnet Python** : `(252, 105)` et `(1071, 462)`.

In [9]:
// Illustration 3 — Exercice 3 : trouver le τ d'Euclide (à compléter)
IReadOnlyList<(int A, int B)> TrajectoireEuclideExercice(int a, int b)
{
    // TODO etudiant (Etape 1)
    return Array.Empty<(int A, int B)>();
}

IReadOnlyList<int> SuiteTauEuclideExercice(int a, int b)
{
    // TODO etudiant (Etape 2)
    return Array.Empty<int>();
}

// Couples cités du carnet Python, cellule 22.
Console.WriteLine("Exercice à compléter : trajectoire d'Euclide et suite des τ pour (252, 105) et (1071, 462) (couples cités du carnet Python).");

Exercice à compléter : trajectoire d'Euclide et suite des τ pour (252, 105) et (1071, 462) (couples cités du carnet Python).


### Exercice 4 — Collatz, le cas limite : mesurer l'absence de τ

La conjecture de Collatz (`n` pair `↦ n/2` ; `n` impair `↦ 3n+1`) résiste précisément parce que **personne ne connaît de τ qui décroisse strictement le long d'une trajectoire** — sa terminaison est un problème ouvert. Mesurez cette absence : pour la trajectoire de `n = 27` (une des longues), suivez les trois candidats naturels — la **valeur** `n`, le **nombre de bits** de `n`, et `ω(n)` (diviseurs premiers distincts, comme au §3.2) — et comptez les pas où chacun **augmente strictement**.

In [10]:
// Illustration 4 — Exercice 4 : Collatz n = 27, trajectoire et mesures candidates (à compléter)
IReadOnlyList<int> TrajectoireCollatzExercice(int n)
{
    // TODO etudiant (Etape 1)
    return Array.Empty<int>();
}

int OmegaPetitExercice(int v)
{
    // TODO etudiant (Etape 2)
    return 0;
}

// Entrée citée du carnet Python, cellule 24.
const int collatzExerciseStart = 27;
Console.WriteLine(Inv($"Exercice à compléter : trajectoire de n = {collatzExerciseStart} (entrée citée du carnet Python), compter les pas croissants de n, du nombre de bits et de ω(n)."));

Exercice à compléter : trajectoire de n = 27 (entrée citée du carnet Python), compter les pas croissants de n, du nombre de bits et de ω(n).


### 3.3 Test de limite — quand τ ne transporte plus

On porte `τ` sur SAT, où aucun ordre n'est donné d'avance. On construit une formule à
3 variables, on descend l'affectation `x₁, x₂, x₃`, et on **mesure la trajectoire** de
deux `τ` naturelles au lieu de supposer ce qu'elles font. Puis on regarde **2-SAT**,
où le problème, lui, *fournit* l'ordre.

In [11]:
// 3.3 — Test de limite : porter τ sur SAT, où aucun ordre n'est donné d'avance.
// On fixe x₁, x₂, x₃ dans cet ordre et on regarde ce que chaque candidat τ
// fait RÉELLEMENT le long de la descente — au lieu de le supposer.
const string sectionSat = "τ — candidats sur SAT";
const string sectionTwoSat = "τ — composantes de 2-SAT";
const int satVariableCount = 3;

// Formule et affectations partielles citées de la cellule 26 du carnet Python.
var satFormula = new[]
{
    new[] { 1, 2, 3 }, new[] { -1, 2, -3 }, new[] { 1, -2, 3 },
    new[] { -1, -2, -3 }, new[] { 1, 2, -3 }
};
var satAssignments = new[]
{
    new Dictionary<int, bool>(),
    new Dictionary<int, bool> { [1] = true },
    new Dictionary<int, bool> { [1] = true, [2] = true },
    new Dictionary<int, bool> { [1] = true, [2] = true, [3] = true }
};

// Une clause est comptée si aucun littéral déjà affecté ne la rend vraie.
int CountNotYetSatisfied(IEnumerable<int[]> clauses, IReadOnlyDictionary<int, bool> assignment) =>
    clauses.Count(clause => !clause.Any(literal =>
        assignment.TryGetValue(Math.Abs(literal), out bool value) && value == (literal > 0)));

void ShowSatCandidate(string label, int[] trajectory, bool[] strictSteps, bool allStrict,
    int[] citedTrajectory, bool[] citedStrictSteps, bool citedAllStrict)
{
    string trajectoryText = FormatIntegerSequence(trajectory);
    string strictText = FormatBooleanSequence(strictSteps);
    Console.WriteLine(label);
    Console.WriteLine($"  trajectoire       : {trajectoryText}");
    Console.WriteLine(Inv($"  décroît strictement ? {strictText}  →  {allStrict}"));
    CheckCondition(sectionSat,
        $"{label} — trajectoire calculée {trajectoryText}, référence Python citée {FormatIntegerSequence(citedTrajectory)}",
        trajectory.SequenceEqual(citedTrajectory));
    CheckCondition(sectionSat,
        $"{label} — pas stricts calculés {strictText}, référence Python citée {FormatBooleanSequence(citedStrictSteps)}",
        strictSteps.SequenceEqual(citedStrictSteps));
    CheckCondition(sectionSat, $"{label} — tous les pas sont stricts", allStrict, citedAllStrict);
}

// Trajectoires, pas stricts et verdicts attendus cités de la sortie Python, cellule 26.
int[] tauA = satAssignments.Select(assignment => CountNotYetSatisfied(satFormula, assignment)).ToArray();
bool[] strictA = StrictDecreaseFlags(tauA);
bool allStrictA = strictA.All(step => step);
ShowSatCandidate("candidat A : τ = clauses non encore satisfaites", tauA, strictA, allStrictA,
    new[] { 5, 2, 1, 1 }, new[] { true, true, false }, false);
CheckCondition(sectionSat, "Le dernier pas de τ_A est plat", tauA[^1] == tauA[^2]);
Console.WriteLine("  Le dernier pas est PLAT : τ_A décroît, mais pas strictement.");
Console.WriteLine("  Elle ne peut donc pas fonder la terminaison par descente stricte.");
Console.WriteLine();

int[] tauB = satAssignments.Select(assignment => satVariableCount - assignment.Count).ToArray();
bool[] strictB = StrictDecreaseFlags(tauB);
bool allStrictB = strictB.All(step => step);
ShowSatCandidate("candidat B : τ = variables non encore affectées", tauB, strictB, allStrictB,
    new[] { 3, 2, 1, 0 }, new[] { true, true, true }, true);
Console.WriteLine("  Elle décroît strictement — mais QUOI QU'IL ARRIVE : elle ne lit jamais F.");
Console.WriteLine("  Elle borne la profondeur de l'énumération, mais ne dit pas si une affectation satisfait F.");

// Sur 2-SAT, en revanche, le problème FOURNIT l'ordre : les composantes
// fortement connexes du graphe d'implication, et leur condensation est un DAG.
Dictionary<int, int> StrongComponents(int[] vertices, Dictionary<int, List<int>> successors)
{
    // Kosaraju, première passe : construire l'ordre de fin des visites.
    var visited = new HashSet<int>();
    var finishOrder = new List<int>();
    foreach (int start in vertices)
    {
        if (!visited.Add(start))
            continue;
        var path = new Stack<(int Vertex, int NextIndex)>();
        path.Push((start, 0));
        while (path.Count > 0)
        {
            var frame = path.Pop();
            var neighbours = successors[frame.Vertex];
            if (frame.NextIndex == neighbours.Count)
            {
                finishOrder.Add(frame.Vertex);
                continue;
            }
            path.Push((frame.Vertex, frame.NextIndex + 1));
            int next = neighbours[frame.NextIndex];
            if (visited.Add(next))
                path.Push((next, 0));
        }
    }

    // Deuxième passe : transposer le graphe et parcourir l'ordre de fin inverse.
    var predecessors = vertices.ToDictionary(vertex => vertex, _ => new List<int>());
    foreach (var entry in successors)
        foreach (int next in entry.Value)
            predecessors[next].Add(entry.Key);

    var component = new Dictionary<int, int>();
    int componentId = 0;
    for (int i = finishOrder.Count - 1; i >= 0; i--)
    {
        int start = finishOrder[i];
        if (!component.TryAdd(start, componentId))
            continue;
        var pending = new Stack<int>();
        pending.Push(start);
        while (pending.Count > 0)
        {
            int vertex = pending.Pop();
            foreach (int previous in predecessors[vertex])
                if (component.TryAdd(previous, componentId)) // Un seul numéro pour TOUT le bloc.
                    pending.Push(previous);
        }
        componentId++;
    }
    return component;
}

(bool Satisfiable, int ComponentCount) DecideTwoSat((int Left, int Right)[] clauses, int variableCount)
{
    var variables = Enumerable.Range(1, variableCount).ToArray();
    var vertices = variables.SelectMany(variable => new[] { variable, -variable }).ToArray();
    var successors = vertices.ToDictionary(vertex => vertex, _ => new List<int>());
    // (a ∨ b) donne les deux implications ¬a → b et ¬b → a.
    foreach (var (left, right) in clauses)
    {
        successors[-left].Add(right);
        successors[-right].Add(left);
    }
    var component = StrongComponents(vertices, successors);
    bool satisfiable = variables.All(variable => component[variable] != component[-variable]);
    return (satisfiable, component.Values.Distinct().Count());
}

// Instances et résultats attendus cités du carnet Python, cellule 26.
var twoSatCases = new (string Label, (int Left, int Right)[] Clauses, int VariableCount,
    int CitedComponents, bool CitedSatisfiable)[]
{
    ("(x₁ ∨ ¬x₂) ∧ (¬x₁ ∨ x₂) ∧ (x₁ ∨ x₂)", new[] { (1, -2), (-1, 2), (1, 2) }, 2, 2, true),
    ("(x₁ ∨ x₁) ∧ (¬x₁ ∨ ¬x₁)", new[] { (1, 1), (-1, -1) }, 1, 1, false)
};
Console.WriteLine("\n2-SAT : le problème fournit l'ordre (condensation du graphe d'implication)");
foreach (var instance in twoSatCases)
{
    var result = DecideTwoSat(instance.Clauses, instance.VariableCount);
    Console.WriteLine(Inv($"  {instance.Label,-38} → {result.ComponentCount} composante(s), satisfiable = {result.Satisfiable}"));
    CheckNear(sectionTwoSat, $"{instance.Label} — composantes (référence Python citée)",
        result.ComponentCount, instance.CitedComponents, ExactIntegerTolerance);
    CheckCondition(sectionTwoSat, $"{instance.Label} — satisfiable (référence Python citée)",
        result.Satisfiable, instance.CitedSatisfiable);
}
Console.WriteLine("\n  τ n'est pas une formule qu'on transporte : c'est un patron qu'on");
Console.WriteLine("  CONSTRUIT à partir d'une structure que le problème doit fournir.");
Console.WriteLine("  En 2-SAT, cette structure donne un critère de décision ;");
Console.WriteLine("  les deux τ naturelles ci-dessus échouent chacune à leur façon à fournir un tel critère.");

// Bilan des contrôles réellement passés : les comparateurs lèvent à la première divergence.
var expectedParitySections = new[]
{
    sectionRuzsa, sectionLimiteRuzsa, sectionAleatoireRuzsa, sectionTriangle,
    sectionProjection, sectionText, sectionOmega, sectionSat, sectionTwoSat
};
var checksBySection = parityChecks.ToLookup(check => check.Section, StringComparer.Ordinal);
var minimumSignChecks = checksBySection[sectionAleatoireRuzsa]
    .Where(check => check.Label.EndsWith(" : min d ≥ 0", StringComparison.Ordinal)).ToArray();
if (minimumSignChecks.Length != ruzsaOperations.Length ||
    !ruzsaOperations.All(law => minimumSignChecks.Count(check => check.Label == $"{law.Name} : min d ≥ 0") == 1))
    throw new InvalidOperationException("Le bilan exige un contrôle de signe réussi pour chacune des quatre opérations.");

Console.WriteLine("\n=== Attestation de parité C# / Python ===");
foreach (string section in expectedParitySections)
{
    var checks = checksBySection[section].ToArray();
    if (checks.Length == 0)
        throw new InvalidOperationException($"Comparaisons manquantes pour {section}.");
    if (section == sectionAleatoireRuzsa)
        Console.WriteLine(Inv($"  {section} : {minimumSignChecks.Length}/{ruzsaOperations.Length} signes conformes ; {checks.Length} contrôles réussis (signes et effectifs)."));
    else
        Console.WriteLine(Inv($"  {section} : {checks.Length} contrôles réussis ; écart absolu maximal enregistré = {checks.Max(check => check.Error):G6}."));
}

candidat A : τ = clauses non encore satisfaites


  trajectoire       : [5, 2, 1, 1]


  décroît strictement ? [True, True, False]  →  False


  CONFORME — candidat A : τ = clauses non encore satisfaites — trajectoire calculée [5, 2, 1, 1], référence Python citée [5, 2, 1, 1] : observé = True, attendu = True


  CONFORME — candidat A : τ = clauses non encore satisfaites — pas stricts calculés [True, True, False], référence Python citée [True, True, False] : observé = True, attendu = True


  CONFORME — candidat A : τ = clauses non encore satisfaites — tous les pas sont stricts : observé = False, attendu = False


... (28 outputs left out)

  Projection / fibres — texte brut : 4 contrôles réussis ; écart absolu maximal enregistré = 0.000439345.


  τ — factorisation : 4 contrôles réussis ; écart absolu maximal enregistré = 0.


  τ — candidats sur SAT : 7 contrôles réussis ; écart absolu maximal enregistré = 0.


  τ — composantes de 2-SAT : 4 contrôles réussis ; écart absolu maximal enregistré = 0.


## 4. Friction naturelle — ce qui a résisté pendant la digestion

La digestion des trois primitives n'a pas été linéaire. Trois obstacles ont rythmé le travail, et les nommer ici est le contenu réel de cette section — pas un préambule.

### 4.1 Pourquoi Ruzsa a résisté au test (a)

La cellule du test 1.3(a) ne mesure pas Ruzsa « sur son terrain ». Elle le mesure **sur le même ensemble** `{0, 1, 2, 3}`, avec **les mêmes distributions** `{0: 0.5, 1: 0.5}`, sous **deux lois de groupe différentes** — `ℤ/4ℤ` cyclique et `(ℤ/2ℤ)²` Klein. La difficulté rencontrée : `d` change de valeur quand la loi change, alors que distributions et ensemble sont fixés. Ce **n'est pas** un artefact numérique ; c'est une propriété de Ruzsa, et l'obstacle pédagogique est précisément d'**exhiber cette dépendance** sans tomber dans le piège inverse (« la valeur de `d` est invariante par changement de groupe »). Le testeur novice qui verrait `d=0` pour `ℤ/4ℤ` et `d=0` pour Klein conclurait à l'invariance — c'est exactement l'erreur que la cellule défait.

La leçon : sur Ruzsa, le test de limite n'a de valeur que **mesuré**, pas **annoncé**.

### 4.2 Pourquoi la décomposition projection/fibres cède sur le texte brut

La cellule du test 2.3 ne mesure pas la décomposition sur un vrai couple `(X, Y)` probabilisé. Elle **tente** la décomposition sur un texte brut où `Y` n'a pas de statut probabiliste. La difficulté rencontrée : `H(text)` après normalisation **a un sens**, mais `H(X | Y)` **n'en a pas** — il n'y a pas de σ-algèbre canonique sur « mots ». L'obstacle pédagogique est de **reconnaître cette absence** sans conclure que la décomposition est fausse : elle est **incomplète**, pas invalide. Construire une mesure rendrait la décomposition légitime, mais ce n'est pas le propos de la cellule.

La leçon : sur la décomposition projection/fibres, **la limite est une absence de mesure**.

### 4.3 Pourquoi τ a cédé sur SAT, doublement

Le test de limite sur SAT mesure **deux** candidats τ, puis examine 2-SAT. La première (`τ = clauses non encore satisfaites`) **descend mais pas strictement** — trajectoire `[5, 2, 1, 1]`, dernier pas plat. La seconde (`τ = variables non encore affectées`) **descend strictement** mais **ne lit jamais la formule** — elle prouve la terminaison de l'énumération, sans fournir un critère de satisfaction. Laquelle des deux « échoue » ? Les deux, mais **chacune à sa manière**. Le test naïf « τ décroît ⇒ ça marche » est invalide : il faut distinguer **lit** (la fonctionnelle lit-elle la structure du problème ?) et **décroît** (descend-elle bien ?).

## 5. Chemin de découverte — ce qui distingue une digestion d'une reconstruction

### 5.1 Pourquoi aller vers ANALYSE-04, pas rester sur ANALYSE-03

ANALYSE-03 contient la preuve formelle PFR dans le lac `pfr` (https://github.com/teorth/pfr). L'investigation qui a mené à ANALYSE-04 est partie d'un constat : la preuve est **instructive en acte**, mais **pas en patron**. Le pattern « entropie ⇒ structure algébrique » y est **utilisé** mais pas **extrait**. Aller vers ANALYSE-04 a consisté à **isoler trois primitives candidates** dans la preuve — Ruzsa, projection/fibres, τ — et à **mesurer leur transportabilité** hors de leur terrain. Sans cette mesure, ANALYSE-04 n'aurait pas de contenu propre : il aurait été une relecture d'ANALYSE-03.

### 5.2 Essais ratés utiles

- **L'essai « appliquons Ruzsa à un dataset ML »** (notes d'étudiants) a échoué parce que les notes ne sont pas des variables aléatoires sur un groupe additif. Cet échec a forcé à **déplacer le test de limite de Ruzsa vers l'algébrique** (`ℤ/4ℤ` vs Klein) plutôt que vers l'applicatif, où la limite est invisible.
- **L'essai « mesurons τ sur factorisation »** donne un témoin positif, pas un test de limite : même le dernier pas, d'un nombre premier à 1, fait décroître `ω`. Pour **chercher un problème où τ n'a pas d'ordre naturel**, le test de limite passe à SAT.
- **L'essai « inversons la lecture : lisons la conclusion d'abord »** (trois primitives, ranger par transportabilité) a échoué parce que la lecture de la preuve formelle ne donne **pas** l'inventaire des primitives : elle les utilise sans les nommer.

### 5.3 Dette résiduelle

- **PFR polynomial généralisé** : la preuve étudiée ici porte sur `F₂ⁿ`. Le carnet n'étudie pas le passage à d'autres groupes.
- **Décomposition projection/fibres sur ICT** : la primitive 2 est notée comme « probablement la plus pertinente » pour ICT (information mutuelle `I(X;Y)` analogue à entropie conditionnelle), mais la mesure canonique **n'est pas construite**.
- **τ sur problèmes non-combinatoires** : la primitive 3 a été testée sur SAT et 2-SAT, tous deux combinatoires. Son transport à un problème continu (par exemple, terminaison d'un schéma itératif continu) n'est pas mesuré.

### 5.4 Transmission

Le notebook est conçu pour un lecteur ayant déjà vu ANALYSE-03. La section 4 doit être lue **après** les tests de limite, pas avant : la friction n'a de sens qu'**une fois la mesure exhibée**.

### Verdict Primitive 3

**Transportable sous condition** : `τ_strictly_decreases` exige un **ordre bien fondé** sous-jacent (typiquement `ℕ` ou un ordinal). Le test de limite sur SAT montre que le transport échoue de **deux façons distinctes** : la `τ` qui lit la formule ne décroît **pas strictement** (trajectoire plate au dernier pas), et celle qui décroît strictement **ne lit jamais la formule** — elle prouve que l'énumération s'arrête, sans fournir un critère de satisfaction. Sur **2-SAT** en revanche la structure existe (condensation du graphe d'implication) et la décision suit.

**Subtilité** : `τ` est un **patron**, pas une formule figée. Pour chaque problème, on doit **construire** la `τ` appropriée — et c'est souvent **l'étape créative** de la preuve. Dans PFR, `τ` est construit à partir de l'entropie ; dans ANALYSE-03, cellule 6, c'est l'invariant qui garantit que le blueprint termine.

**Leçon** : la primitive est **la moins transportable des trois** parce qu'elle dépend de la structure du problème. Mais quand elle s'applique, elle est **irremplaçable** — c'est elle qui dit « quand la procédure a fini ».

***

## 6. Conclusion — où les primitives cessent de valoir

### Tableau récapitulatif

| # | Primitive | Transport | Condition | Sans la condition |
|---|---|---|---|---|
| 1 | `RuzsaDistance` | **conditionnel** | groupe additif `(G, +)` | `d` peut devenir **négatif** : ce n'est plus une distance |
| 2 | `X = π(X) + (X \| π(X))` | **large** | distribution de probabilité | décomposition vide (projection non définie) |
| 3 | `τ_strictly_decreases` | **patron** | ordre bien-fondé | terminaison non garantie |

Chaque primitive a été **appliquée hors de son terrain** et le notebook **exhibe** — par une cellule qui s'exécute, pas par un paragraphe qui l'annonce — l'endroit exact où elle cesse de valoir. C'est cette honnêteté qui distingue une « digestion » d'une « analogie ».

### Lien vers ANALYSE-03 et le lac `pfr`

ANALYSE-03 contient les **preuves formelles** dans le lac `pfr` (https://github.com/teorth/pfr). ANALYSE-04 ne les reproduit pas — il **extrait les patrons** et **vérifie leur portée**. Les trois primitives ci-dessus sont les **briques élémentaires** que la preuve PFR utilise pour passer de la compression à la structure algébrique.

### Vers ICT (sans mapping affine)

> PFR ne « prouve » rien pour ICT. Mais il fournit un **exemple certifié** où une quantité informationnelle **force** une structure algébrique. C'est un patron, pas une analogie : quand on dispose d'une « monnaie commune » (entropie) entre deux théories, et qu'on voit la même structure forcée par les mêmes contraintes, on a une **primitive transportable**, pas une ressemblance.

Pour ICT, la primitive la plus pertinente est **probablement la 2** (décomposition projection/fibres) : l'information mutuelle `I(X;Y)` y joue un rôle analogue à l'entropie conditionnelle. Mais on doit **construire explicitement la mesure**, pas plaquer la formule.

### Honnêteté finale

> Les trois primitives sont des **outils**, pas des **lois**. Leur transportabilité est **conditionnelle** et le test de limite fait partie du livrable. Le carnet ANALYSE-04 a nommé **où** chacune cesse de valoir — c'est le contenu pédagogique réel.

> Provenance : [Version Python](ANALYSE-04-PFR-Primitives-Python.ipynb).